# MOOC 02 — Résultats et investigations

But : apprendre à lire les résultats existants et à distinguer les métriques qui valident vraiment la thèse de celles qui peuvent masquer le signal.

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
RUN = ROOT / "runs" / "smoke"
HISTORY = RUN / "history.json"
print(HISTORY)

## 1. Lire `history.json`

Chaque élément de `history.json` correspond à un round terminé. Les clés sont plates : `val/*`, `rollout/*`, `pool/*`, `ddpm/*`, `timing/*`, etc.

In [ ]:
history = json.loads(HISTORY.read_text())
print("rounds", len(history))
print("keys in last round", len(history[-1]))
for key in sorted(history[-1])[:60]:
    print(key)

In [ ]:
KEYS = [
    "train/loss",
    "ddpm/loss",
    "pool/loss_mean",
    "pool/loss_p90",
    "val/nrmse_mean",
    "val/nrmse_p90",
    "rollout/nrmse_mean",
    "rollout/nrmse_final",
]

for row in history:
    print("round", row["round"])
    for key in KEYS:
        if key in row:
            print(f"  {key:24s} {row[key]:.6g}")

## 2. Courbes essentielles

En général :

- `val/nrmse_mean` mesure le one-step moyen sur validation uniforme ;
- `rollout/nrmse_final` mesure la stabilité autoregressive ;
- `pool/loss_*` décrit la difficulté du pool courant ;
- les `hard_val/*` sont les métriques centrales pour les états rares/tube.

In [ ]:
def series(key):
    xs, ys = [], []
    for row in history:
        if key in row and row[key] is not None and math.isfinite(row[key]):
            xs.append(row["round"])
            ys.append(row[key])
    return np.array(xs), np.array(ys)

plot_keys = [
    "train/loss",
    "ddpm/loss",
    "val/nrmse_mean",
    "rollout/nrmse_final",
    "pool/loss_mean",
    "pool/loss_p90",
]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, key in zip(axes.ravel(), plot_keys):
    x, y = series(key)
    ax.plot(x, y, marker="o")
    ax.set_title(key)
    ax.set_xlabel("round")
    ax.grid(True, alpha=0.3)
plt.tight_layout()

## 3. Rapport automatique du run local

Cette cellule produit un mini-rapport relatif entre le premier et le dernier round.

In [ ]:
first, last = history[0], history[-1]
for key in ["val/nrmse_mean", "rollout/nrmse_mean", "rollout/nrmse_final", "pool/loss_mean", "pool/loss_p90"]:
    if key not in first or key not in last:
        continue
    a, b = first[key], last[key]
    change = 100 * (b - a) / (abs(a) + 1e-12)
    print(f"{key:24s} {a:.6g} -> {b:.6g} ({change:+.1f}%)")

## 4. Ce que dit le handoff

Le run smoke n'est pas une preuve scientifique ; il vérifie surtout l'intégration. Les conclusions empiriques sérieuses sont dans `HANDOFF.md`.

Résumé opérationnel :

- Phase 2 KS : gros gains sur tube one-step RMSE et TV-hard p99 ;
- bulk uniforme : pas robustement neutre ;
- rollout@100 : instable selon seed ;
- lowamp-hard : pire pour toutes les variantes, résultat négatif à conserver ;
- Burgers Phase 3 : validation invalide à cause d'artefacts solveur ;
- v3 : corrige réalisme, self-feeding, stats de difficulté, steering et audit.

In [ ]:
text = (ROOT / "HANDOFF.md").read_text().splitlines()
sections = ["## What is empirically established", "## The v3 method", "## Immediate next steps", "## Known open issues"]
for section in sections:
    print(f"\n{section}")
    try:
        start = next(i for i, line in enumerate(text) if line.startswith(section))
    except StopIteration:
        print("missing")
        continue
    end = next((i for i in range(start + 1, len(text)) if text[i].startswith("## ")), len(text))
    for line in text[start + 1:end][:18]:
        print(line)

## 5. Hard validation : pourquoi elle est centrale

Les validations uniformes peuvent cacher les gains, car elles sont dominées par les régions faciles/fréquentes. Les scripts importants :

- `scripts/build_hard_validation.py` : queues TV, low amplitude, coverage ;
- `scripts/build_tube_validation.py` : états off-attractor perturbés ;
- `scripts/build_diverse_validation.py` : version diversifiée et stratifiée ;
- `scripts/eval_hard_posthoc.py` : évaluation finale sur checkpoints ;
- `scripts/eval_rollout_posthoc.py` : horizon stable `K_tau`.

In [ ]:
for rel in [
    "scripts/build_hard_validation.py",
    "scripts/build_tube_validation.py",
    "scripts/build_diverse_validation.py",
    "scripts/eval_hard_posthoc.py",
    "scripts/eval_rollout_posthoc.py",
]:
    path = ROOT / rel
    print(f"\n--- {rel} ---")
    lines = path.read_text().splitlines()
    marker = chr(34) * 3
    in_doc = False
    shown = 0
    for line in lines:
        if marker in line:
            in_doc = not in_doc
            continue
        if in_doc and shown < 12:
            print(line[:120])
            shown += 1

## 6. Lire une campagne multi-seed

Le script `scripts/analyze_campaign.py` suppose des dossiers du type `<arm>_seed<seed>/history.json` et compare les derniers rounds complets.

Arms connus : `uniform_baseline`, `noise_inject`, `random_tube`, `tube_select`, `mined_ic`, `gen_v3`, `gen_v3_edit`, etc.

In [ ]:
analysis = ROOT / "scripts" / "analyze_campaign.py"
text = analysis.read_text()
for needle in ["ARMS =", "SEEDS =", "SETS =", "BULK =", "def collect", "def main"]:
    line = next(i for i, line in enumerate(text.splitlines(), start=1) if needle in line)
    print(f"{needle:14s} -> {analysis}:{line}")

## 7. Commandes post-hoc à adapter

Remplace les chemins par ceux de ta campagne. La règle : comparer à round/budget apparié, et garder les baselines `random_tube` / `mined_ic` pour tester si le générateur appris ajoute vraiment quelque chose.

In [ ]:
commands = [
    ".venv/bin/python scripts/eval_hard_posthoc.py --runs <RUNS...> --uniform <uniform.npz> --hard <hard_dir> --baseline uniform_baseline",
    ".venv/bin/python scripts/eval_rollout_posthoc.py --runs <RUNS...> --uniform <uniform.npz> --steps 100 --baseline uniform_baseline",
    ".venv/bin/python scripts/analyze_campaign.py --base-dir <campaign_dir> --metric nrmse_p50 --vs uniform_baseline,random_tube",
]
for cmd in commands:
    print(cmd)

## Exercice

Sur une campagne réelle, fais une table à quatre colonnes :

1. bulk uniforme (`val/nrmse_mean`) ;
2. hard TV (`hard_val/val_hard_tv_div/*`) ;
3. tube (`hard_val/val_tube_*/*`) ;
4. rollout (`K_tau` ou `rollout/nrmse_final`).

Conclusion acceptable seulement si le gain tient sur hard/tube et bat `random_tube` à budget apparié.